# 01 — Data Exploration

CMU Keystroke Dynamics Benchmark (`DSL-StrongPasswordData.csv`): authentic human typing baselines for UEBA Policy Engine evaluation.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_dataset, list_subjects, get_subject_data
from src.features.keystroke import get_feature_columns

df = load_dataset()
subjects = list_subjects(df)
print(f'{df.shape[0]} rows × {df.shape[1]} cols | {len(subjects)} subjects')
df.head()

In [ ]:
# Hold-time profile for a representative subject (human baseline)
subject = 's002'
subj = get_subject_data(df, subject)
hold_cols = get_feature_columns('hold')
hold_means = subj[hold_cols].mean()

plt.figure(figsize=(10, 4))
hold_means.plot(kind='line', marker='o', color='teal')
plt.title(f'Mean Key Hold Times — {subject}')
plt.ylabel('Time (s)')
plt.xlabel('Key')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Flight-time (DD) distributions across a few subjects
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, sid in zip(axes, subjects[:3]):
    s = get_subject_data(df, sid)
    flight = s[get_feature_columns('flight')].mean(axis=1)
    sns.histplot(flight, ax=ax, bins=30, color='steelblue')
    ax.set_title(sid)
    ax.set_xlabel('Mean flight time (s)')
fig.suptitle('Per-repetition mean flight time')
fig.tight_layout()
plt.show()

In [ ]:
# Session variance supports continuous-monitoring narrative
sess = subj.groupby('sessionIndex')[hold_cols].mean().mean(axis=1)
plt.figure(figsize=(6, 3))
sess.plot(kind='bar', color='coral')
plt.title(f'{subject}: mean hold time by session')
plt.ylabel('Time (s)')
plt.xlabel('Session')
plt.tight_layout()
plt.show()
print(subj.groupby('sessionIndex').size())